# BLOQUE 05 – Riesgo Territorial y Entorno
## Objetivo
Integrar las capas de la previa etapa en un solo geopackage, y realizar el filtrado de lo que se necesitará en la etapa 3.
## Inputs
- 1_D5C1_Económico.gpkg
- 1_D5C2_Equipamientos.gpkg
- 1_D5C3_Movilidad.gpkg
- 1_D5C4_Estructura_urbana.gpkg
## Procesos
- Integración
- Filtrado
## Output
- 2_Dimensiones.gpkg
    - 2_D5C1_ue.shp
    - 2_D5C1_sinergia_ue.shp
    - 2_D5C2_sinergia_eq.shp
    - 2_D5C3_Infraestructura_peatonal_mzn.shp
    - 2_D5C4_delitos.shp
    - 2_D5C4_inundaciones.shp

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
ruta_economico = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D5C1_Económico.gpkg"
ruta_equipamientos = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D5C2_Equipamientos.gpkg"
ruta_movilidad = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D5C3_Movilidad.gpkg"
ruta_riesgos = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D5C4_Riesgos.gpkg"

ruta_salida = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_2\2_Dimensiones.gpkg"

# Carga de datos

In [3]:
gpd.list_layers(ruta_equipamientos)

,name,geometry_type
0,1_D5C2V1_equipamientos,Point
1,1_D5C2V1_equipamientos_Educación,Point
2,1_D5C2V1_equipamientos_Salud,Point
3,1_D5C2V1_equipamientos_Recreación,Point
4,1_D5C2V1_equipamientos_Servicios,Point
5,1_D5C2V1_equipamientos_Comercio,Point
6,1_D5C2V1_equipamientos_Cultura,Point
7,1_D5C2V1_equipamientos_mzn,Polygon


In [4]:
ue = gpd.read_file(ruta_economico, layer = "1_D5C1V1_Unidades_económicas")
eq = gpd.read_file(ruta_equipamientos, layer = "1_D5C2V1_equipamientos")
mov = gpd.read_file(ruta_movilidad, layer = "1_D5C3V1_Infraestructura_peatonal_mzn")
delitos = gpd.read_file(ruta_riesgos, layer = "1_D5C4V1_delitos")
inundaciones = gpd.read_file(ruta_riesgos, layer = "1_D5C4V1_inundaciones")

In [5]:
ue.columns

Index(['id', 'nom_estab', 'codigo_act', 'nombre_act', 'per_ocu', 'cve_ent',
       'cve_mun', 'cve_loc', 'ageb', 'manzana', 'fecha_alta', 'Num_Emp',
       'geometry'],
      dtype='object')

In [6]:
eq.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'CONDICION', 'GEOGRAFICO', 'NOMSERV', 'TIPO', 'CATEGORIA', 'RADIO',
       'IMP', 'geometry'],
      dtype='object')

In [7]:
mov.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'AMBITO', 'TIPOMZA', 'ARB', 'ALU', 'BAN', 'geometry'],
      dtype='object')

In [8]:
delitos.columns

Index(['fecha', 'delito', 'clave_mun', 'hora', 'Categoría', 'geometry'], dtype='object')

In [9]:
inundaciones.columns

Index(['Vulnerabil', 'geometry'], dtype='object')

# filtrado estado y municipios

In [10]:
# EXTRAER JALISCO
ue_ent = ue[ue['cve_ent'] == '14'].copy()
eq_ent = eq[eq['CVE_ENT'] == '14'].copy()
mov_ent = mov[mov['CVE_ENT'] == '14'].copy()

# Lista de municipios AMG
municipios = ['002','039', '044', '051','070','097', '098', '101', '120', '124']

# Filtrar el DataFrame solo con los municipios de interés
ue_mun = ue_ent[
    (ue_ent['cve_mun'].isin(municipios)) & 
    (ue_ent['cve_ent'] == '14')
]

eq_mun = eq_ent[
    (eq_ent['CVE_MUN'].isin(municipios)) & 
    (eq_ent['CVE_ENT'] == '14')
]

mov_mun = mov_ent[
    (mov_ent['CVE_MUN'].isin(municipios)) & 
    (mov_ent['CVE_ENT'] == '14')
]

delitos_mun = delitos[delitos['clave_mun'].isin(municipios)]

# Extracción de elementos de interés

## Unidades económicas

In [11]:
actividades = ue_mun['nombre_act'].unique()

search_terms = ['físi', 'gym']

filtered_unique = [
    val for val in actividades
    if any(term.lower() in val.lower() for term in search_terms)
]
filtered_unique

['Servicios de levantamiento geofísico',
 'Consultorios del sector privado de audiología y de terapia ocupacional, física y del lenguaje',
 'Consultorios del sector público de audiología y de terapia ocupacional, física y del lenguaje',
 'Centros de acondicionamiento físico del sector privado',
 'Centros de acondicionamiento físico del sector público']

In [12]:
actividades_interes = [
    'Centros de acondicionamiento físico del sector privado',
 'Centros de acondicionamiento físico del sector público']

ue_sinergia_mun = ue_mun[ue_mun['nombre_act'].isin(actividades_interes)].copy()

## Equipamientos

In [13]:
eq_mun['GEOGRAFICO'].unique()

array(['Plaza', 'Instalación Deportiva o Recreativa', 'Pozo', 'Templo',
       'Centro de Asistencia Médica', 'Tanque de Agua', 'Escuela',
       'Instalación Gubernamental', 'Mercado', 'Cementerio',
       'Instalación Diversa', 'Centro Comercial',
       'Instalación de Comunicación', 'Subestación Eléctrica',
       'Instalación de Servicios'], dtype=object)

In [14]:
equipamientos = eq_mun[eq_mun['GEOGRAFICO'] == 'Centro de Asistencia Médica']['TIPO'].unique()
equipamientos

array(['Centro de Salud', 'Hospital', 'Centro de Rehabilitación'],
      dtype=object)

In [15]:
eq_sinergia_mun = eq_mun[(eq_mun['GEOGRAFICO'] == 'Centro de Asistencia Médica') & (eq_mun['TIPO'].isin(['Centro de Salud', 'Hospital']))]

## delitos

In [16]:
# delitos_mun['delito'].unique()

In [17]:
# actividades_interes = [
#     'Robo a negocio'
# ]

# delitos_mun = delitos_mun[delitos_mun['delito'].isin(actividades_interes)].copy()

# Limpieza asteriscos y n/d

In [18]:
# Reemplazar * con 1
def replace(dataframe):
    return dataframe.replace('*', '1')

# Limpiar *
ue_mun = replace(ue_mun)
ue_sinergia_mun = replace(ue_sinergia_mun)
eq_sinergia_mun = replace(eq_sinergia_mun)
mov_mun = replace(mov_mun)
delitos_mun = replace(delitos_mun)
inundaciones = replace(inundaciones)

# Reemplazar N/D
ue_mun = ue_mun.replace('N/D', np.nan)
ue_sinergia_mun = ue_sinergia_mun.replace('N/D', np.nan)
eq_sinergia_mun = eq_sinergia_mun.replace('N/D', np.nan)
mov_mun = mov_mun.replace('N/D', np.nan)
delitos_mun = delitos_mun.replace('N/D', np.nan)
inundaciones = inundaciones.replace('N/D', np.nan)

# Actualizar valores sin información de "null" a 0
ue_mun = ue_mun.fillna(0)
ue_sinergia_mun = ue_sinergia_mun.fillna(0)
eq_sinergia_mun = eq_sinergia_mun.fillna(0)
mov_mun = mov_mun.fillna(0)
delitos_mun = delitos_mun.fillna(0)
inundaciones = inundaciones.fillna(0)

# Guardado 

In [19]:
ue_mun.to_file(ruta_salida, layer='2_D5C1_ue', driver='GPKG')
ue_sinergia_mun.to_file(ruta_salida, layer='2_D5C1_sinergia_ue', driver='GPKG')
eq_sinergia_mun.to_file(ruta_salida, layer='2_D5C2_sinergia_eq', driver='GPKG')
mov_mun.to_file(ruta_salida, layer='2_D5C3_Infraestructura_peatonal_mzn', driver='GPKG')
delitos_mun.to_file(ruta_salida, layer='2_D5C4_delitos', driver='GPKG')
inundaciones.to_file(ruta_salida, layer='2_D5C4_inundaciones', driver='GPKG')